In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base = "/content/drive/MyDrive"

print("Searching for earthquake project folders...\n")

for root, dirs, files in os.walk(base):
    depth = root[len(base):].count(os.sep)
    if depth > 4:
        dirs[:] = []
        continue

    for d in dirs:
        if "quake_project" in d.lower() or d == "v4_fresh":
            print(os.path.join(root, d))

Searching for earthquake project folders...

/content/drive/MyDrive/quake_project_v4
/content/drive/MyDrive/quake_project_v4/results/v4_fresh
/content/drive/MyDrive/quake_project_v4/results/quake_project_v4_clean (1)
/content/drive/MyDrive/quake_project_v4/results/quake_project_v4_clean (1)/quake_project_v4


In [ ]:
from pathlib import Path
import shutil

MAIN = Path("/content/drive/MyDrive/quake_project_v4")

# Exact obsolete SCEDC files
targets = [
    MAIN / "raw_data/scedc_socal_m25.parquet",
    MAIN / "raw_data/scedc_socal_m25.meta.json",
    MAIN / "dataset/scedc_socal_m25_labeled.parquet",
    MAIN / "dataset/scedc_socal_m25_features.parquet",

    # Summary/provenance files must be regenerated
    MAIN / "dataset/data_provenance.json",
    MAIN / "dataset/dataset_summary.csv",
    MAIN / "dataset/label_counts.csv",
    MAIN / "dataset/feature_availability.csv",
]

deleted = []
not_found = []

# Delete exact files
for p in targets:
    if p.exists():
        p.unlink()
        deleted.append(str(p))
    else:
        not_found.append(str(p))

# Delete old SCEDC download chunks only
chunk_dir = MAIN / "raw_data/chunks"
if chunk_dir.exists():
    for p in chunk_dir.glob("scedc_*.parquet"):
        p.unlink()
        deleted.append(str(p))

# Delete old SCEDC dataset checkpoints only
ckpt_dir = MAIN / "dataset/ckpt"
if ckpt_dir.exists():
    for p in ckpt_dir.glob("scedc_socal_m25_*"):
        if p.is_dir():
            shutil.rmtree(p)
        else:
            p.unlink()
        deleted.append(str(p))

print("=" * 70)
print("SCEDC CLEANUP COMPLETE")
print("=" * 70)

print(f"\nDeleted: {len(deleted)} items")
for p in deleted:
    print("DELETED:", p)

print("\nImportant files that MUST still exist:")
important = [
    MAIN / "raw_data/comcat_global_m4.parquet",
    MAIN / "dataset/comcat_global_m4_features.parquet",
    MAIN / "results/v4_fresh",
]

for p in important:
    print(("OK   " if p.exists() else "MISSING"), p)

print("\nNo analysis has been run yet.")

SCEDC CLEANUP COMPLETE

Deleted: 72 items
DELETED: /content/drive/MyDrive/quake_project_v4/raw_data/scedc_socal_m25.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/raw_data/scedc_socal_m25.meta.json
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/scedc_socal_m25_labeled.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/scedc_socal_m25_features.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/data_provenance.json
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/dataset_summary.csv
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/label_counts.csv
DELETED: /content/drive/MyDrive/quake_project_v4/dataset/feature_availability.csv
DELETED: /content/drive/MyDrive/quake_project_v4/raw_data/chunks/scedc_1981.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/raw_data/chunks/scedc_1982.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/raw_data/chunks/scedc_1983.parquet
DELETED: /content/drive/MyDrive/quake_project_v4/r

In [ ]:
%cd /content/drive/MyDrive/quake_project_v4

!python code/make_dataset.py

/content/drive/MyDrive/quake_project_v4
log          -> /content/drive/MyDrive/quake_project_v4/dataset/logs/dataset_20260921T143843Z.txt
project root : /content/drive/MyDrive/quake_project_v4
catalog end  : 2026-09-01 (pinned)
seed         : 42

[stage 1] download and clean catalogs   (+0.0 min)
[1/3] comcat_global_m4 present
[2/3] SCEDC, 1981 onwards, M>=2.5, 32.0-37.0N -122.0--114.0E (no /count endpoint, so windows split reactively)
  endpoint format: text
  1981-01-01 -> 1982-01-01:    627
  1982-01-01 -> 1983-01-01:   1102
  1983-01-01 -> 1984-01-01:   1494
  1984-01-01 -> 1985-01-01:   1025
  1985-01-01 -> 1986-01-01:    855
  1986-01-01 -> 1987-01-01:   1852
  1987-01-01 -> 1988-01-01:   2009
  1988-01-01 -> 1989-01-01:   1234
  1989-01-01 -> 1990-01-01:   1161
  1990-01-01 -> 1991-01-01:    757
  1991-01-01 -> 1992-01-01:    634
  1992-01-01 -> 1993-01-01:   6119
  1993-01-01 -> 1994-01-01:    926
  1994-01-01 -> 1995-01-01:   1641
  1995-01-01 -> 1996-01-01:    714
  1996-01-0

In [ ]:
%cd /content/drive/MyDrive/quake_project_v4

!python code/run_analysis.py \
    --root /content/drive/MyDrive/quake_project_v4 \
    --run-name v5_socal_corrected \
    --gpu

[Errno 2] No such file or directory: '/content/drive/MyDrive/quake_project_v4'
/content
python3: can't open file '/content/code/run_analysis.py': [Errno 2] No such file or directory


In [3]:
%cd /content/drive/MyDrive/quake_project_v4

!pip install -q -r requirements.txt

print("\nInstallation finished.")

# Verify the important packages
import optuna
import catboost
import xgboost
import sklearn

print("Optuna:", optuna.__version__)
print("CatBoost:", catboost.__version__)
print("XGBoost:", xgboost.__version__)
print("scikit-learn:", sklearn.__version__)

/content/drive/MyDrive/quake_project_v4

Installation finished.
Optuna: 5.0.0
CatBoost: 1.2.10
XGBoost: 3.4.1
scikit-learn: 1.6.1


In [4]:
%cd /content/drive/MyDrive/quake_project_v4

!python code/run_analysis.py \
    --root /content/drive/MyDrive/quake_project_v4 \
    --run-name v5_socal_corrected \
    --gpu

/content/drive/MyDrive/quake_project_v4
--gpu given but no GPU was found; continuing on CPU
log        -> /content/drive/MyDrive/quake_project_v4/results/v5_socal_corrected/logs/analysis_20260922T063613Z.txt
root       : /content/drive/MyDrive/quake_project_v4
run name   : v5_socal_corrected
seed       : 42
GPU        : False
dataset    : built 2026-09-21T14:55:27, catalog end 2026-09-01, hashes match
             comcat_global_m4         545220 events  sha 86c669896b6b
             scedc_socal_m25           45648 events  sha a50678e5ccc9
             comcat_himalaya_m4         4710 events  sha 1bbb7828a54d

[stage 1] train models   (+0.1 min)
=== comcat_global_m4: n=545220, train 327132 / val 54522 / calib 81783 / test 81783, 11 features ===
=== scedc_socal_m25: n=45648, train 27388 / val 4565 / calib 6847 / test 6848, 5 features ===
=== comcat_himalaya_m4: n=4710, train 2826 / val 471 / calib 706 / test 707, 11 features ===

[stage 2] calibration audit   (+0.3 min)
  comcat_global_m4

In [5]:
%cd /content/drive/MyDrive/quake_project_v4

!python code/audit_outputs.py \
    --root /content/drive/MyDrive/quake_project_v4 \
    --run-name v5_socal_corrected

/content/drive/MyDrive/quake_project_v4
auditing /content/drive/MyDrive/quake_project_v4/results/v5_socal_corrected

dataset
  [ ok  ] comcat_global_m4: feature file matches provenance
  [ ok  ] scedc_socal_m25: feature file matches provenance
  [ ok  ] comcat_himalaya_m4: feature file matches provenance

tables
  [ ok  ] 20 of 20 expected tables present

figures
  [ ok  ] 14 figures present

models
  [ ok  ] comcat_global_m4: trained and recalibrated
  [ ok  ] scedc_socal_m25: trained and recalibrated
  [ ok  ] comcat_himalaya_m4: trained and recalibrated

conformal sets
  [ ok  ] set-size rates sum to one
  [ ok  ] mean set size matches the identity

ETAS
  [ ok  ] comcat_global_m4: branching ratio 0.715, calibration matched
  [ ok  ] scedc_socal_m25: branching ratio 0.947, calibration not_converged
  [ ok  ] comcat_himalaya_m4: branching ratio 0.950, calibration stability_limited
  [ ok  ] 15 realisations over 3 catalogs

manifest
  [ ok  ] manifest records seed, environment, code h